# Missing Data 
Missing values are encoded as NaN. Most ML algorithms can't use them directly.

1. Missingness mechanisms: MCAR, MAR, MNAR (and what each does to the data)
2. Detect missing values and patterns
3. Simple handling: drop rows/columns, mean, median, mode, forward/backward fill
4. Model-based imputation: regression, KNN, missing indicators
5. Compare imputers (imputation error + downstream model error)
6. Takeaways

In [ ]:
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

os.makedirs('plots', exist_ok=True)
rng = np.random.default_rng(0)

## 1. Data and the three missingness mechanisms
Features: age, education, income (skewed, depends on age/education). We hide
values of **income** in three ways:
* MCAR: every row has the same 30% chance of being missing
* MAR: older people are more likely to hide income (depends on observed *age*)
* MNAR: high earners hide income (depends on the *missing value itself*)

In [ ]:
n = 1500
age = rng.normal(40, 10, n)
edu = rng.normal(14, 2.5, n)
income = np.exp(0.02 * age + 0.12 * edu + rng.normal(0, 0.25, n) + 7.0) / 1000  # in k$, right-skewed
X_full = np.c_[age, edu, income]
names = ["age", "education", "income"]

def sigmoid(z): return 1 / (1 + np.exp(-z))

def calibrate(score, frac=0.30):
    """Shift logits so that the average missing probability equals frac."""
    lo, hi = -20, 20
    for _ in range(60):
        mid = (lo + hi) / 2
        if sigmoid(score + mid).mean() > frac: hi = mid
        else: lo = mid
    return sigmoid(score + (lo + hi) / 2)

def zs(v): return (v - v.mean()) / v.std()

p_mcar = np.full(n, 0.30)
p_mar = calibrate(2.0 * zs(age))
p_mnar = calibrate(2.0 * zs(income))
masks = {"MCAR": rng.random(n) < p_mcar, "MAR": rng.random(n) < p_mar, "MNAR": rng.random(n) < p_mnar}

print(f"True mean income: {income.mean():.2f}k  median: {np.median(income):.2f}k")
for k, m in masks.items():
    print(f"{k}: {m.mean():.1%} missing | mean of OBSERVED income = {income[~m].mean():.2f}k "
          f"| mean age of rows with missing income = {age[m].mean():.1f} (overall {age.mean():.1f})")

Under MCAR the observed mean is unbiased. Under MAR the observed income is biased
only through age (fixable by conditioning on age). Under MNAR the observed mean is
biased downward and the data alone can't tell us that - the hardest case.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8), sharex=True)
bins = np.linspace(income.min(), np.percentile(income, 99.5), 40)
for a, (k, m) in zip(ax, masks.items()):
    a.hist(income, bins, alpha=.4, label="all", density=True)
    a.hist(income[~m], bins, alpha=.5, label="observed", density=True)
    a.set_title(f"{k}: observed vs true income"); a.set_xlabel("income (k$)"); a.legend()
plt.tight_layout(); plt.savefig('plots/01_mechanisms.png', dpi=110); plt.close()

## 2. Detect missing values and patterns
Equivalent of `df.isnull().sum()` + a missingness matrix.

In [ ]:
def make_missing(mask_income, extra_age_frac=0.10, seed=1):
    """Return a copy of X_full with NaNs in income (given mask) and ~10% MCAR in education."""
    r = np.random.default_rng(seed)
    X = X_full.copy()
    X[mask_income, 2] = np.nan
    X[r.random(n) < extra_age_frac, 1] = np.nan
    return X

Xm = make_missing(masks["MAR"])
print("Missing counts per column:", dict(zip(names, np.isnan(Xm).sum(0))))
print("Missing % per column     :", dict(zip(names, np.round(np.isnan(Xm).mean(0) * 100, 1))))
print("Rows with >=1 missing    :", np.isnan(Xm).any(1).sum(), "of", n)

fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(np.isnan(Xm[:120]).T, aspect='auto', cmap='Greys', interpolation='nearest')
ax.set_yticks(range(3)); ax.set_yticklabels(names); ax.set_xlabel("row (first 120)")
ax.set_title("Missingness matrix (black = missing)")
plt.tight_layout(); plt.savefig('plots/02_missing_pattern.png', dpi=110); plt.close()

## 3. Simple handling
Remove rows/columns, or fill with mean / median / mode, and forward / backward fill
for ordered (time-series) data.

In [ ]:
def drop_rows(X): return X[~np.isnan(X).any(1)]
def drop_cols(X, thresh=0.5): return X[:, np.isnan(X).mean(0) < thresh]

def mean_impute(X, stats=None):
    X = X.copy(); stats = np.nanmean(X, 0) if stats is None else stats
    r, c = np.where(np.isnan(X)); X[r, c] = stats[c]; return X, stats

def median_impute(X, stats=None):
    X = X.copy(); stats = np.nanmedian(X, 0) if stats is None else stats
    r, c = np.where(np.isnan(X)); X[r, c] = stats[c]; return X, stats

def mode_impute(labels, missing_token=None):
    """Categorical mode imputation (missing = None)."""
    vals = [v for v in labels if v is not missing_token]
    cats, cnt = np.unique(vals, return_counts=True); mode = cats[np.argmax(cnt)]
    return [mode if v is missing_token else v for v in labels], mode

def ffill(x):
    x = x.copy()
    for i in range(1, len(x)):
        if np.isnan(x[i]): x[i] = x[i - 1]
    return x

def bfill(x):
    return ffill(x[::-1])[::-1]

print("drop rows keeps", len(drop_rows(Xm)), "of", n, "rows;  drop cols (>50% missing) keeps", drop_cols(Xm).shape[1], "cols")
print("\nDoc examples:")
print("mean of [20,25,30,?,35]  ->", np.nanmean([20, 25, 30, np.nan, 35]))
print("median of [20,22,24,25,100,?] ->", np.nanmedian([20, 22, 24, 25, 100, np.nan]))
print("mode of [Red,Blue,Blue,Green,?] ->", mode_impute(["Red", "Blue", "Blue", "Green", None])[1])
print("ffill [20,22,?,?,25] ->", ffill(np.array([20, 22, np.nan, np.nan, 25.])))
print("bfill [20,?,?,25]    ->", bfill(np.array([20, np.nan, np.nan, 25.])))

Mean imputation shrinks the variance and distorts correlations (every filled value sits
at the same point). Median is more robust for skewed income.

In [ ]:
Xmean, _ = mean_impute(Xm); Xmed, _ = median_impute(Xm)
miss_inc = np.isnan(Xm[:, 2])
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8), sharex=True, sharey=True)
for a, (t, Xi) in zip(ax, [("Truth", X_full), ("Mean imputed", Xmean), ("Median imputed", Xmed)]):
    a.scatter(Xi[~miss_inc, 0], Xi[~miss_inc, 2], s=5, alpha=.4, label="observed")
    a.scatter(Xi[miss_inc, 0], Xi[miss_inc, 2], s=5, c='r', alpha=.5, label="imputed/hidden")
    a.set_title(t); a.set_xlabel("age"); a.set_ylabel("income")
ax[0].legend(); plt.tight_layout(); plt.savefig('plots/03_simple_imputation.png', dpi=110); plt.close()
print("std of income  true: %.2f | mean-imputed: %.2f | median-imputed: %.2f" %
      (income.std(), Xmean[:, 2].std(), Xmed[:, 2].std()))

Time-series fill: a noisy sine signal with a gap.

In [ ]:
t = np.arange(60); sig = np.sin(t / 6) * 10 + 20 + rng.normal(0, .3, 60)
sig_m = sig.copy(); sig_m[20:28] = np.nan; sig_m[45:48] = np.nan
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(t, sig, 'k:', label="truth"); ax.plot(t, ffill(sig_m), label="ffill"); ax.plot(t, bfill(sig_m), label="bfill")
ax.plot(t, sig_m, 'ko', ms=3, label="observed"); ax.legend(); ax.set_title("Forward vs backward fill")
plt.tight_layout(); plt.savefig('plots/04_ffill_bfill.png', dpi=110); plt.close()

## 4. Predict missing values
**Regression imputation**: fit linear regression of the incomplete column on the other
columns using complete rows, then predict. **Stochastic** version adds residual noise to
preserve variance. **KNN imputation**: average the target column over the k nearest rows
(distance on commonly-observed, standardised features).

In [ ]:
def lin_fit(A, y):
    A1 = np.c_[np.ones(len(A)), A]; return np.linalg.lstsq(A1, y, rcond=None)[0]
def lin_pred(w, A): return np.c_[np.ones(len(A)), A] @ w

def regression_impute(X, col, stochastic=False, rng=rng, log_target=True):
    """Impute X[:,col] from the other columns. Other columns first mean-filled."""
    X = X.copy(); others = [j for j in range(X.shape[1]) if j != col]
    Z, _ = mean_impute(X[:, others]); m = np.isnan(X[:, col])
    y = X[~m, col]; f = np.log if log_target else (lambda v: v); fi = np.exp if log_target else (lambda v: v)
    w = lin_fit(Z[~m], f(y)); res = f(y) - lin_pred(w, Z[~m])
    pred = lin_pred(w, Z[m])
    if stochastic: pred = pred + rng.normal(0, res.std(), m.sum())
    X[m, col] = fi(pred)
    X, _ = mean_impute(X)   # other columns with gaps (e.g. education) get mean-filled
    return X

def knn_impute(X, k=7):
    X = X.copy(); mu = np.nanmean(X, 0); sd = np.nanstd(X, 0); Zs = (X - mu) / sd
    out = X.copy()
    for i in np.where(np.isnan(X).any(1))[0]:
        for c in np.where(np.isnan(X[i]))[0]:
            donors = np.where(~np.isnan(X[:, c]))[0]; donors = donors[donors != i]
            shared = [j for j in range(X.shape[1]) if j != c and not np.isnan(Zs[i, j])]
            D = np.nanmean((Zs[donors][:, shared] - Zs[i, shared]) ** 2, axis=1) if shared else np.zeros(len(donors))
            D = np.where(np.isnan(D), np.inf, D)   # donors lacking all shared features are last
            nn = donors[np.argsort(D)[:k]]; out[i, c] = X[nn, c].mean()
    return out

Xreg = regression_impute(Xm, 2); Xregs = regression_impute(Xm, 2, stochastic=True)
Xknn = knn_impute(Xm)
print("Regression impute done; KNN impute done. Remaining NaNs:", np.isnan(Xreg[:, 2]).sum(), np.isnan(Xknn).sum())

**Missing indicators**: append a binary column "was this value missing?". Useful when
missingness itself carries signal (MAR/MNAR), e.g. high earners hiding income.

In [ ]:
def add_indicators(X_orig, X_imp):
    cols = np.where(np.isnan(X_orig).any(0))[0]
    return np.c_[X_imp, np.isnan(X_orig[:, cols]).astype(float)]

## 5. Compare imputers
(a) Imputation error: RMSE on the hidden income values (log scale to tame skew).
(b) Downstream: a target `y` that depends on income; linear regression trained on the
imputed training rows, evaluated on a clean test set. Imputation statistics are fit on
train only (otherwise leakage).
Run for all 3 mechanisms.

In [ ]:
y_target = 0.05 * age + 0.4 * edu + 0.08 * income + rng.normal(0, 1.0, n)
tr = np.arange(n) < 1000; te = ~tr
Xte_clean = X_full[te]

def rmse(a, b): return float(np.sqrt(np.mean((a - b) ** 2)))

def run(mech):
    Xm = make_missing(masks[mech]); hid = masks[mech] & tr
    Xtr_m = Xm[tr]; out = {}
    mean_tr, st = mean_impute(Xtr_m); med_tr, stm = median_impute(Xtr_m)
    cands = {"drop rows": None, "mean": mean_tr, "median": med_tr,
             "regression": regression_impute(Xtr_m, 2), "stoch. regr.": regression_impute(Xtr_m, 2, stochastic=True),
             "KNN": knn_impute(Xtr_m), "mean+indicator": add_indicators(Xtr_m, mean_tr)}
    for name, Xi in cands.items():
        if Xi is None:
            keep = ~np.isnan(Xtr_m).any(1); w = lin_fit(Xtr_m[keep], y_target[tr][keep]); imp_err = np.nan
            test_mse = np.mean((lin_pred(w, Xte_clean) - y_target[te]) ** 2)
        else:
            imp_err = rmse(np.log(Xi[hid[tr], 2]), np.log(income[hid]))
            w = lin_fit(Xi, y_target[tr])
            Xt = Xte_clean if Xi.shape[1] == 3 else np.c_[Xte_clean, np.zeros((te.sum(), Xi.shape[1] - 3))]
            test_mse = np.mean((lin_pred(w, Xt) - y_target[te]) ** 2)
        out[name] = (imp_err, test_mse)
    return out

results = {m: run(m) for m in masks}
print(f"{'method':16s}" + "".join(f"{m+' impRMSE':>16s}{m+' testMSE':>16s}" for m in masks))
for name in results["MCAR"]:
    print(f"{name:16s}" + "".join(f"{results[m][name][0]:16.3f}{results[m][name][1]:16.3f}" for m in masks))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.2)); meths = list(results["MCAR"]); xi = np.arange(len(meths))
for j, m in enumerate(masks):
    ax[0].bar(xi + (j - 1) * .27, [np.nan_to_num(results[m][k][0]) for k in meths], .27, label=m)
    ax[1].bar(xi + (j - 1) * .27, [results[m][k][1] for k in meths], .27, label=m)
for a, t_ in zip(ax, ["Imputation RMSE (log income, hidden values)", "Downstream test MSE (clean test set)"]):
    a.set_xticks(xi); a.set_xticklabels(meths, rotation=25); a.set_title(t_); a.legend()
plt.tight_layout(); plt.savefig('plots/05_imputer_comparison.png', dpi=110); plt.close()

Distribution check: imputed vs true income for each method (MAR case).

In [ ]:
Xm_mar = make_missing(masks["MAR"]); hid = masks["MAR"]
fig, ax = plt.subplots(1, 1, figsize=(7, 4)); b = np.linspace(0, np.percentile(income, 99.5), 40)
ax.hist(income[hid], b, alpha=.35, density=True, label="true hidden")
for lab, Xi in [("mean", mean_impute(Xm_mar)[0]), ("regression", regression_impute(Xm_mar, 2)),
                ("stochastic regr.", regression_impute(Xm_mar, 2, stochastic=True)), ("KNN", knn_impute(Xm_mar))]:
    ax.hist(Xi[hid, 2], b, histtype='step', lw=1.8, density=True, label=lab)
ax.legend(); ax.set_title("Distribution of imputed income values (MAR)"); ax.set_xlabel("income (k$)")
plt.tight_layout(); plt.savefig('plots/06_imputed_distributions.png', dpi=110); plt.close()

## 6. Takeaways
* Understand *why* data is missing (MCAR / MAR / MNAR) before choosing a method.
* Dropping rows is fine for few MCAR rows; costly otherwise and biased under MAR/MNAR.
* Mean/median are simple but shrink variance and weaken correlations; median for skew.
* Regression / KNN exploit relations between features (good under MAR); stochastic
  regression keeps the variance. MNAR can't be fixed from the observed data alone.
* Add missing indicators when missingness is informative.
* Fit imputation statistics on the training set only (avoid data leakage).

In [ ]:
# Optional sklearn sanity check
try:
    from sklearn.impute import SimpleImputer, KNNImputer
    a = SimpleImputer(strategy='median').fit_transform(Xm); b_ = median_impute(Xm)[0]
    print("median impute matches sklearn:", np.allclose(a, b_))
    a = KNNImputer(n_neighbors=7).fit_transform(Xm)
    print("KNN mean abs diff vs sklearn (different distance handling, expected small):",
          round(float(np.mean(np.abs(a[:, 2] - knn_impute(Xm)[:, 2]))), 3))
except Exception as e:
    print("sklearn check skipped:", e)